# JSON Patch with System.Text.Json

This notebook demonstrates how to use JSON Patch operations with the `System.Text.Json` library in .NET.

JSON Patch is a standard format (RFC 6902) for describing changes to a JSON document.

ASP.NET Core provides built-in support for JSON Patch through the `Microsoft.AspNetCore.JsonPatch.SystemTextJson` package.

## The JSON Patch Playground project

This project demonstrates how to use JSON Patch in an ASP.NET Core Web API application.

It includes examples of PATCH endpoints in both Minimal APIs and Controllers.

To start the project, run the following command in the terminal:


cd web-api/jsonpatch/JsonPatch && dotnet run

In [ ]:
$env:HostAddress = "http://localhost:5221"

## Apply a JSON Patch document to an object

The following examples demonstrate how to use the [ApplyTo(Object)] method to apply a JSON Patch document to an object.

[ApplyTo(Object)]: https://learn.microsoft.com/dotnet/api/microsoft.aspnetcore.jsonpatch.systemtextjson.jsonpatchdocument.applyto

The following example demonstrates:
- The add, replace, and remove operations.
- Operations on nested properties.
- Adding a new item to an array.
- Using a JSON String Enum Converter in a JSON patch document.

In [ ]:
# Get the original customer

curl -s -X GET `
    -H "Accept: application/json" `
    $env:HostAddress/customers/1


In [ ]:
# Update the object with PATCH. The response contains the updated object.

curl -s -X PATCH `
    -H "Content-Type: application/json-patch+json" `
    -H "Accept: application/json" `
    -d '[
      {"op": "replace", "path": "/firstName", "value": "Jane" },
      { "op": "remove", "path": "/email"},
      { "op": "add", "path": "/address/zipCode", "value": "90210" },
      { "op": "add", "path": "/phoneNumbers/-", "value": { "number": "987-654-3210", "type": "Work" } }
    ]' `
    $env:HostAddress/customers/1

The [ApplyTo(Object)] method generally follows the conventions and options of [System.Text.Json] for processing the [JsonPatchDocument\<TModel\>], including the behavior controlled by the following options:

- [JsonNumberHandling]: Whether numeric properties are read from strings.
- [PropertyNameCaseInsensitive]: Whether property names are case-sensitive.

Key differences between [System.Text.Json] and the new [JsonPatchDocument\<TModel\>] implementation:

- The runtime type of the target object, not the declared type, determines which properties [ApplyTo(Object)] patches.
- [System.Text.Json] deserialization relies on the declared type to identify eligible properties.

[ApplyTo(Object)]: https://learn.microsoft.com/dotnet/api/microsoft.aspnetcore.jsonpatch.systemtextjson.jsonpatchdocument.applyto
[JsonPatchDocument\<TModel\>]: https://learn.microsoft.com/dotnet/api/microsoft.aspnetcore.jsonpatch.systemtextjson.jsonpatchdocument-1
[JsonNumberHandling]: https://learn.microsoft.com/dotnet/api/system.text.json.serialization.jsonnumberhandling
[PropertyNameCaseInsensitive]: https://learn.microsoft.com/dotnet/api/system.text.json.jsonserializeroptions.propertynamecaseinsensitive
[System.Text.Json]: https://learn.microsoft.com/dotnet/api/system.text.json

## Apply a JsonPatchDocument with error handling

There are various errors that can occur when applying a JSON Patch document. For example, the target object may not have the specified property, or the value specified might be incompatible with the property type.

JSON Patch supports the test operation, which checks if a specified value equals the target property. If it doesn't, it returns an error.

The following example demonstrates how to handle these errors gracefully.

> &#9432; Important!<br/>
> The object passed to the [ApplyTo(Object)] method is modified in place. The caller is responsible for discarding changes if any operation fails.

[ApplyTo(Object)]: https://learn.microsoft.com/dotnet/api/microsoft.aspnetcore.jsonpatch.systemtextjson.jsonpatchdocument.applyto

In [ ]:
# Attempt to patch a non-existing property

curl -s -X PATCH `
    -H "Content-Type: application/json-patch+json" `
    -H "Accept: application/json" `
    -d '[
      { "op": "add", "path": "/foobar", "value": 42 }
    ]' `
    $env:HostAddress/customers/1

In [ ]:
# PATCH with test operation that fails

curl -s -X PATCH `
    -H "Content-Type: application/json-patch+json" `
    -H "Accept: application/json" `
    -d '[
      { "op": "replace", "path": "/email", "value": "marysmith@gmail.com"},
      { "op": "test", "path": "/firstName", "value": "Mary" },
      { "op": "replace", "path": "/lastName", "value": "Smith" }
    ]' `
    $env:HostAddress/customers/1

Note that the application detected the errors in the PATCH operation and did not apply any of the changes.

In [ ]:
curl -s -X GET `
    -H "Accept: application/json" `
    $env:HostAddress/customers/1